<a href="https://colab.research.google.com/github/SKM1919/liftlens/blob/main/notebooks/01_hillstrom_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-uplift -q

import pandas as pd
from sklift.datasets import fetch_hillstrom

bunch = fetch_hillstrom(target_col='all')
df = pd.concat([bunch.data, bunch.target, bunch.treatment], axis=1)

print(df.shape)
df.head()



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.5 MB/s eta 0:00:00


Hillstrom dataset:   0%|          | 0.00/443k [00:00<?, ?iB/s]

(64000, 12)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,visit,conversion,spend,segment
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,0,0,0.0,Womens E-Mail
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,0,0,0.0,No E-Mail
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,0,0,0.0,Womens E-Mail
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,0,0,0.0,Mens E-Mail
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,0,0,0.0,Womens E-Mail


In [ ]:
print(df['segment'].value_counts())

df.groupby('segment')[['visit', 'conversion', 'spend']].mean()

segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64


,visit,conversion,spend
segment,,,
Mens E-Mail,0.182757,0.012531,1.422617
No E-Mail,0.106167,0.005726,0.652789
Womens E-Mail,0.151400,0.008837,1.077202


## Key findings: Hillstrom email experiment

- **Randomisation check:** The three groups are almost equal in size (~21,300 each), so the random split worked and the comparison is fair.
- **Baseline:** Without any email, 10.6% of customers visited the site and 0.57% made a purchase.
- **Mens E-Mail** had the biggest effect: +7.7 percentage points in visits, +0.68 pp in conversions, and +$0.77 spend per customer. For every 1,000 customers emailed, that's about 77 extra visits, 7 extra purchases, and $770 extra revenue.
- **Womens E-Mail** also worked, but less: +4.5 pp visits, +0.31 pp conversions, +$0.42 spend per customer.

**Business takeaway:** Both emails drive real incremental revenue on average, and Mens performs best. But 10.6% of customers visit even without an email, so emailing everyone wastes budget on people who would have come anyway. Next step: test whether these lifts are statistically significant, then find *which* customers the email actually persuades.

## Is the lift statistically significant?

In [ ]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

control = df[df['segment'] == 'No E-Mail']
results = []

for email in ['Mens E-Mail', 'Womens E-Mail']:
    treat = df[df['segment'] == email]

    for metric in ['visit', 'conversion']:
        # successes and group sizes
        x_t, n_t = treat[metric].sum(), len(treat)
        x_c, n_c = control[metric].sum(), len(control)

        # is the difference real?
        z, p = proportions_ztest([x_t, x_c], [n_t, n_c])

        # 95% range for the true lift
        low, high = confint_proportions_2indep(x_t, n_t, x_c, n_c, method='wald')

        results.append({
            'email': email,
            'metric': metric,
            'lift_pp': round((x_t/n_t - x_c/n_c) * 100, 2),
            'ci_low_pp': round(low * 100, 2),
            'ci_high_pp': round(high * 100, 2),
            'z': round(z, 2),
            'p_value': p,
            'significant': p < 0.05
        })

pd.DataFrame(results)


,email,metric,lift_pp,ci_low_pp,ci_high_pp,z,p_value,significant
0,Mens E-Mail,visit,7.66,7.00,8.32,22.49,5.685165e-112,True
1,Mens E-Mail,conversion,0.68,0.50,0.86,7.39,1.523224e-13,True
2,Womens E-Mail,visit,4.52,3.89,5.16,13.95,3.182403e-44,True
3,Womens E-Mail,conversion,0.31,0.15,0.47,3.78,1.571051e-04,True


## Power and Minimum Detectable Effect (MDE)



In [ ]:
from scipy.stats import norm
import numpy as np

def mde(p_base, n_per_group, alpha=0.05, power=0.80):
    """Smallest lift (as a proportion) we can reliably detect."""
    z_alpha = norm.ppf(1 - alpha / 2)   # 1.96 for 95% confidence
    z_power = norm.ppf(power)           # 0.84 for 80% power
    se = np.sqrt(2 * p_base * (1 - p_base) / n_per_group)
    return (z_alpha + z_power) * se

def sample_size(p_base, lift, alpha=0.05, power=0.80):
    """Customers needed PER GROUP to detect a given lift."""
    z_alpha = norm.ppf(1 - alpha / 2)
    z_power = norm.ppf(power)
    return int(np.ceil(2 * p_base * (1 - p_base) * ((z_alpha + z_power) / lift) ** 2))

n = len(control)
rows = []
for metric in ['visit', 'conversion']:
    p0 = control[metric].mean()
    m = mde(p0, n)
    rows.append({
        'metric': metric,
        'baseline_%': round(p0 * 100, 2),
        'n_per_group': n,
        'MDE_pp': round(m * 100, 2),
        'MDE_relative_%': round(m / p0 * 100, 1)
    })

pd.DataFrame(rows)

,metric,baseline_%,n_per_group,MDE_pp,MDE_relative_%
0,visit,10.62,21306,0.84,7.9
1,conversion,0.57,21306,0.20,35.8


In [ ]:
p0 = control['conversion'].mean()
for lift_pp in [0.5, 0.3, 0.2, 0.1]:
    print(f"To detect a +{lift_pp} pp conversion lift: "
          f"{sample_size(p0, lift_pp / 100):,} customers per group")


To detect a +0.5 pp conversion lift: 3,575 customers per group
To detect a +0.3 pp conversion lift: 9,931 customers per group
To detect a +0.2 pp conversion lift: 22,344 customers per group
To detect a +0.1 pp conversion lift: 89,373 customers per group


### Summary: Power and MDE

- With ~21,300 customers per group, this experiment could reliably detect a visit lift of **0.84 pp** or a conversion lift of **0.20 pp** (80% power, 95% confidence). Every observed lift cleared this bar, but Womens/conversion (0.31 pp) only just did, which explains its wide confidence interval.
- **Halving the effect you want to detect needs 4× the sample:** a 0.2 pp conversion lift needs ~22,000 customers per group, while 0.1 pp needs ~89,000.
- **Scenario:** with a 50,000-customer list, I would not run a test aiming to detect a 0.1 pp conversion lift, because it would likely come back "not significant" even if the subject line works. Instead, I'd use visits as the primary metric, combine results across several sends, or test a bolder change.